# 08 - Operational Monitoring Examples

## Purpose

Create simple audit tables and examples that help a Fabric team monitor pipeline runs, row counts, freshness, and data quality results.

This notebook is intentionally lightweight so beginners can understand the pattern before adopting enterprise monitoring tooling.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_timestamp = datetime.now(timezone.utc)
batch_id = run_timestamp.strftime("%Y%m%d%H%M%S")

audit_pipeline_table = "audit_pipeline_run"
audit_quality_table = "audit_data_quality_result"

print(f"Monitoring batch ID: {batch_id}")

In [ ]:
# Create audit tables if they do not exist.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_pipeline_table} (
    batch_id STRING,
    environment STRING,
    pipeline_name STRING,
    activity_name STRING,
    status STRING,
    started_at TIMESTAMP,
    ended_at TIMESTAMP,
    rows_read LONG,
    rows_written LONG,
    message STRING
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_quality_table} (
    batch_id STRING,
    table_name STRING,
    rule_name STRING,
    severity STRING,
    status STRING,
    failed_count LONG,
    run_timestamp TIMESTAMP
)
USING DELTA
""")

In [ ]:
# Write a sample audit record for the current Lakehouse state.
entities = ["customers", "accounts", "products", "branches", "transactions"]
rows = []
for entity in entities:
    bronze_name = f"bronze_{entity}"
    silver_name = f"silver_{entity}"
    if spark.catalog.tableExists(bronze_name) and spark.catalog.tableExists(silver_name):
        rows.append((
            batch_id,
            "dev",
            "pl_transform_retail_banking_medallion",
            f"reconcile_{entity}",
            "Succeeded",
            run_timestamp,
            datetime.now(timezone.utc),
            spark.table(bronze_name).count(),
            spark.table(silver_name).count(),
            "Bronze to Silver reconciliation sample"
        ))

audit_df = spark.createDataFrame(rows, ["batch_id", "environment", "pipeline_name", "activity_name", "status", "started_at", "ended_at", "rows_read", "rows_written", "message"])
audit_df.write.format("delta").mode("append").saveAsTable(audit_pipeline_table)
display(audit_df)

In [ ]:
# Write sample quality status from required Gold tables.
required_gold_tables = ["dim_customer", "dim_account", "dim_product", "dim_branch", "dim_date", "fact_transaction"]
quality_rows = []
for table_name in required_gold_tables:
    exists = spark.catalog.tableExists(table_name)
    row_count = spark.table(table_name).count() if exists else 0
    quality_rows.append((
        batch_id,
        table_name,
        "table_exists_and_has_rows",
        "critical",
        "PASS" if exists and row_count > 0 else "FAIL",
        0 if exists and row_count > 0 else 1,
        run_timestamp
    ))

quality_df = spark.createDataFrame(quality_rows, ["batch_id", "table_name", "rule_name", "severity", "status", "failed_count", "run_timestamp"])
quality_df.write.format("delta").mode("append").saveAsTable(audit_quality_table)
display(quality_df)

In [ ]:
# Monitoring views for engineers.
latest_runs = (
    spark.table(audit_pipeline_table)
    .withColumn("duration_seconds", F.unix_timestamp("ended_at") - F.unix_timestamp("started_at"))
    .orderBy(F.col("started_at").desc())
)
display(latest_runs)

quality_summary = (
    spark.table(audit_quality_table)
    .groupBy("batch_id", "status", "severity")
    .agg(F.count("*").alias("rule_count"), F.sum("failed_count").alias("failed_count"))
    .orderBy(F.col("batch_id").desc(), "severity", "status")
)
display(quality_summary)

## Production Considerations

- Store audit tables in a governed operations Lakehouse or schema.
- Add pipeline run IDs, activity IDs, and error details.
- Add alerting for failed quality checks and stale data.
- Build a small operational Power BI report for data owners and engineers.